In [1]:
# Vérifier la version de Python et la disponibilité du GPU
import sys
print(f"Python : {sys.version}")

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "⚠️ Pas de GPU détecté, on utilisera le CPU (plus lent)")

Python : 3.12.6 | packaged by conda-forge | (main, Sep 22 2024, 14:16:49) [GCC 13.3.0]
Fri Mar  6 15:17:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 6000 Ada Gene...    Off |   00000000:AC:00.0 Off |                  Off |
| 30%   36C    P8              9W /  300W |     113MiB /  49140MiB |      0%      Default |
|                                         |          

In [2]:
import torch

print("=== Diagnostic CUDA ===\n")

# Vérifier la version de PyTorch installée
print(f"PyTorch version       : {torch.__version__}")

# La clé : est-ce que PyTorch voit le GPU ?
cuda_dispo = torch.cuda.is_available()
print(f"CUDA disponible       : {cuda_dispo}")

if cuda_dispo:
    print(f"Version CUDA PyTorch  : {torch.version.cuda}")
    print(f"GPU détecté           : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM disponible       : {vram:.1f} GB")
    print("\n✅ Tout est parfait, vous pouvez utiliser device='cuda' sans problème.")
else:
    print("\n⚠️  PyTorch ne voit pas le GPU.")
    print("Cela signifie que PyTorch a été installé en version CPU-only.")
    print("Solution : réinstaller PyTorch avec le support CUDA 12.x")

ModuleNotFoundError: No module named 'torch'

In [3]:
import sys
import subprocess

# Voir quel Python et quel environnement sont utilisés
print(f"Python utilisé : {sys.executable}")

# Voir les environnements Conda disponibles
result = subprocess.run(['conda', 'env', 'list'], capture_output=True, text=True)
print(result.stdout)

Python utilisé : /opt/tljh/user/bin/python3.12
# conda environments:
#
base                     /opt/tljh/user




In [6]:
# --user installe dans ~/local/lib/python3.12/site-packages
# ce qui ne nécessite pas de droits admin
# cu128 = build compilée pour CUDA 12.8, compatible avec votre driver 13.0
!pip install torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu128 \
    --user --quiet

print("✅ Installation terminée — redémarrez maintenant le kernel !")
print("   Kernel → Restart Kernel dans le menu JupyterLab")

✅ Installation terminée — redémarrez maintenant le kernel !
   Kernel → Restart Kernel dans le menu JupyterLab


In [1]:
import torch

print(f"PyTorch version    : {torch.__version__}")
print(f"CUDA disponible    : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU                : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM disponible    : {vram:.1f} GB")
    print("\n🎉 Parfait ! Vous êtes prêt à utiliser SAM 3 sur GPU.")
else:
    print("\n⚠️  CUDA toujours non détecté après installation.")
    print("Copiez ce message complet et partagez-le pour diagnostiquer.")

PyTorch version    : 2.10.0+cu128
CUDA disponible    : True
GPU                : NVIDIA RTX 6000 Ada Generation
VRAM disponible    : 47.4 GB

🎉 Parfait ! Vous êtes prêt à utiliser SAM 3 sur GPU.


In [2]:
# Ultralytics est le package qui intègre SAM 3 nativement
# C'est la façon la plus simple et la plus stable d'utiliser SAM 3
!pip install ultralytics>=8.3.237 --user --quiet

print("✅ Ultralytics installé — redémarrez le kernel une dernière fois !")
print("   Kernel → Restart Kernel dans le menu JupyterLab")

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
✅ Ultralytics installé — redémarrez le kernel une dernière fois !
   Kernel → Restart Kernel dans le menu JupyterLab


In [1]:
import torch
from ultralytics import SAM

# Forcer explicitement le GPU — on ne laisse aucune chance au CPU
device = "cuda"

# Charger SAM 3 Large — le modèle le plus puissant disponible
# Ultralytics téléchargera automatiquement les poids (~2 GB) au premier lancement
model = SAM("sam3_l.pt")
model.to(device)

# Confirmer que le modèle tourne bien sur GPU
print(f"✅ SAM 3 Large chargé sur : {next(model.model.parameters()).device}")
print(f"   VRAM utilisée maintenant : {torch.cuda.memory_allocated(0) / 1024**3:.1f} GB")
print(f"   VRAM encore disponible   : {torch.cuda.memory_reserved(0) / 1024**3:.1f} GB")
print("\n🚀 Vous êtes prêt à segmenter vos images maritimes !")

WARNING ⚠️ user config directory '/home/jupyter-p2m/.config/Ultralytics' is not writable, using '/tmp/Ultralytics'. Set YOLO_CONFIG_DIR to override.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
requirements: Ultralytics requirement ['timm'] not found, attempting AutoUpdate...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/2.6 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━ 2.1/2.6 MB 11.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 11.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/612.9 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 12.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━

FileNotFoundError: [Errno 2] No such file or directory: 'sam3_l.pt'

In [ ]:
# huggingface_hub est la bibliothèque officielle de Hugging Face
# qui permet de télécharger des fichiers depuis leurs serveurs
# en gérant automatiquement la reprise en cas d'interruption
!pip install huggingface_hub --user --quiet

from huggingface_hub import hf_hub_download
import os

# Collez votre token ici entre les guillemets
token = "5JmpVzgyiTCddfJ5lBQYDSDt1F8CnP31GANL"

print("⏳ Téléchargement de SAM 3 en cours (~2.5 GB)... soyez patient")

chemin_poids = hf_hub_download(
    repo_id   = "facebook/sam3",
    filename  = "sam3.pt",
    token     = token,
    local_dir = ".",
)

print(f"✅ Poids téléchargés avec succès !")
print(f"   Chemin : {chemin_poids}")
print(f"   Taille : {os.path.getsize(chemin_poids) / 1024**3:.2f} GB")

⏳ Téléchargement de SAM 3 en cours (~2.5 GB)... soyez patient


sam3.pt:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

✅ Poids téléchargés avec succès !
   Chemin : sam3.pt
   Taille : 3.21 GB


In [3]:
import torch
from ultralytics.models.sam import SAM3SemanticPredictor

# On vérifie une dernière fois que le GPU est bien disponible
# avant de charger le modèle — on ne veut absolument pas
# que SAM 3 tourne sur CPU par inadvertance
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device détecté : {device}")

if device == "cpu":
    raise RuntimeError("⛔ GPU non détecté ! Vérifiez votre installation CUDA.")

# Configuration du predictor SAM 3 en mode segmentation sémantique
# half=True active le mode FP16 (demi-précision) qui divise par 2
# l'utilisation mémoire sans perte significative de précision —
# parfait pour votre RTX 6000 avec 48 GB de VRAM
overrides = dict(
    conf  = 0.25,       # seuil de confiance minimum pour garder une détection
    task  = "segment",  # on veut de la segmentation, pas de la détection simple
    mode  = "predict",  # mode inférence (pas entraînement)
    model = "sam3.pt",  # chemin vers le fichier qu'on vient de télécharger
    half  = True,       # FP16 pour maximiser les performances sur RTX 6000
)

predictor = SAM3SemanticPredictor(overrides=overrides)

# Vérifier que le modèle occupe bien la VRAM du GPU
vram_utilisee = torch.cuda.memory_allocated(0) / 1024**3
vram_totale   = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"✅ SAM 3 chargé avec succès sur {torch.cuda.get_device_name(0)} !")
print(f"   VRAM utilisée     : {vram_utilisee:.1f} GB")
print(f"   VRAM encore libre : {vram_totale - vram_utilisee:.1f} GB")
print(f"\n🚀 Prêt à segmenter vos images maritimes !")

Device détecté : cuda
✅ SAM 3 chargé avec succès sur NVIDIA RTX 6000 Ada Generation !
   VRAM utilisée     : 0.0 GB
   VRAM encore libre : 47.4 GB

🚀 Prêt à segmenter vos images maritimes !
